# 💎 기능 4. 연구 스페이스 (Gems) - 에이전트 구축 및 스트리밍 (심화)

본 가이드는 **Paper Agent** 프로젝트의 **연구 스페이스 (Gems) 기능** 중 **동적 도구 바인딩 기법, PostgreSQL 세션 체크포인터 설정, 그리고 AI 답변 실시간 스트리밍(Token, Status, Papers의 이원화 구조)의 핵심 동작 원리**를 심도 있게 학습합니다.

외부 모듈에서 임포트하는 방식 대신 **실제 백엔드의 `GemAgent` 클래스와 동적 툴 바인딩 및 스트리밍 전송 로직 전체를 노트북 내부에 직접 구현**하여, LangGraph 기반 멀티턴 영구 세션 에이전트가 조립되는 원리를 철저히 학습합니다.

학습 집중에 방해되는 Mock 폴백 시뮬레이터 및 예외 처리 코드를 제거하고, **실제 서비스에서 돌아가는 핵심 프로덕션 코드 위주**로 간결하게 정리되었습니다.

---

## 📌 심화 학습 핵심 포인트
1. **PostgreSQL 기반 체크포인터** (`AsyncPostgresSaver`)
   - LangGraph에서 대화 스레드별 멀티 턴(Multi-turn) 상태가 데이터베이스 테이블(`checkpoints`)에 영구 보존 및 복원되는 구성을 실습합니다.
2. **동적 도구 바인딩 및 커스텀 시스템 프롬프트**
   - 사용자가 설정한 `db_sources` 범위(생명과학/컴퓨터과학/천문학)와 업로드된 문서 파일 유무에 따라, 에이전트가 실행 시점에 알맞은 검색 툴들을 동적으로 적재하는 원리를 이해합니다.
3. **실시간 스트리밍 답변과 상태 및 출처 분리**
   - 스트림 수행 중 툴의 실행 상태(`type: status`), 답변의 텍스트 조각(`type: token`), 그리고 참고한 출처 목록(`type: papers`)을 구분하여 JSON Line 형태로 전송하는 비동기 제너레이터 구조를 이해합니다.
4. **경로 및 샌드박스 연동 검증**
   - 실제 환경변수가 구동되는 환경에서 에이전트를 조립하고 호출하는 기법을 실습합니다.

## 1단계. 환경 및 백엔드 설정 로드

노트북 실행 위치를 설정하고 백엔드 패키지 경로를 주입합니다. 필요한 라이브러리 및 임베딩 모델 연결을 정의합니다.

In [17]:
import os
import sys
import asyncio
import json
import operator
from pathlib import Path
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Any, cast, AsyncGenerator

# 현재 notebooks/gem_service/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

# 필수 모듈 임포트
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langchain.agents import create_agent, AgentState
from pydantic import BaseModel, Field

# text-embedding-3-large 모델 고정 규격 사용
EMBED_MODEL = "text-embedding-3-large"
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=openai_key
)

# 데이터베이스 연결 정보 설정
pgvector_url = database_url.replace("postgresql+asyncpg://", "postgresql+psycopg_async://")

print("✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!


## 2단계. PostgreSQL 기반 체크포인터 설정 (`AsyncPostgresSaver`)

LangGraph의 대화 기록 영구 저장과 세션 관리는 `AsyncPostgresSaver`를 통해 데이터베이스에 저장됩니다.
여기서는 백엔드 설정에 등록된 `DATABASE_URL`을 이용하여 `psycopg` 연결 풀을 열고 테이블을 구축합니다.

In [18]:
from api.database.config.psycopg_pool import psycopg_pool as chat_psycopg_pool

# 1. psycopg 연결 풀 열기 (비동기)
if chat_psycopg_pool.closed:
    await chat_psycopg_pool.open()
    print("✅ psycopg 연결 풀 개설 완료!")
else:
    print("ℹ️ psycopg 연결 풀이 이미 열려 있습니다.")

# 2. AsyncPostgresSaver를 이용한 체크포인터 선언
checkpointer = AsyncPostgresSaver(cast(Any, chat_psycopg_pool))

# 3. 데이터베이스 테이블 존재 여부 확인 및 멱등(Idempotent) 생성
async with chat_psycopg_pool.connection() as conn:
    cur = await conn.execute(
        "SELECT 1 FROM pg_tables WHERE schemaname='public' AND tablename='checkpoints'"
    )
    exists = await cur.fetchone()
    
if not exists:
    await checkpointer.setup()
    print("✅ checkpoints 테이블을 데이터베이스에 구축했습니다.")
else:
    print("✅ checkpoints 테이블이 이미 존재하며, 에이전트와 연결할 준비가 되었습니다.")

ℹ️ psycopg 연결 풀이 이미 열려 있습니다.
✅ checkpoints 테이블이 이미 존재하며, 에이전트와 연결할 준비가 되었습니다.


## 3단계. GemAgent 구현 및 동적 도구 바인딩

실제 백엔드의 [gem_agent.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/gem_agent.py) 코드를 직접 정의합니다.

1. **`GemAgentState`**: 대화 내역(`messages`) 외에, 여러 도구 노드가 반환하는 논문 출처들을 누적할 수 있도록 `sources` 필드를 병합 리듀서(`operator.add`)로 구성합니다.
2. **동적 파일 검색 도구 (`_make_file_search_tool`)**: 업로드된 Gem 문서에 대해 search를 수행하고 `Command(update=...)`로 상태 채널(`sources`, `messages`)에 정보를 주입합니다.
3. **동적 에이전트 빌드 (`_build_agent` / `_build_stream_agent`)**: 사용자가 명시한 `db_sources` (생명공학/컴퓨터과학/천문학)와 업로드된 문서가 존재할 경우 파일 툴을 컴파일 시점에 동적 탑재하고 에이전트를 선언합니다.
4. **동적 시스템 프롬프트 빌더 (`_build_system_prompt`)**: 사용 가능한 도구 목록과 시스템 프롬프트(페르소나)를 결합하여 맞춤 지침을 LLM에 주입합니다.

In [19]:
from api.common.rag_pipeline import (
    search_bio_papers,
    search_cs_papers,
    search_astronomy_papers,
)

# 1. 에이전트의 대화 상태 스키마 정의
class GemAgentState(AgentState):
    sources: Annotated[list[dict], operator.add]   # 검색된 논문 출처 누적

# 2. db_sources 옵션에 따른 실제 RAG 도구 맵
_TOOL_MAP = {
    "bio": search_bio_papers,
    "cs": search_cs_papers,
    "astronomy": search_astronomy_papers,
}

_TOOL_CALL_DESC = {
    "bio": "생명공학·유전체학 관련 질문 → search_bio_papers",
    "cs": "컴퓨터과학 관련 질문 → search_cs_papers",
    "astronomy": "천문학 관련 질문 → search_astronomy_papers",
}
_FILE_TOOL_DESC = "사용자가 업로드한 파일 내용 검색 → search_gem_files (업로드 파일에서 답을 찾을 수 있을 때 반드시 호출)"

# 3. Gem별 업로드 파일 검색용 동적 툴 빌더
def _make_file_search_tool(gem_id: str):
    from api.v1.gems.file_rag import gem_file_rag

    @tool
    async def search_gem_files(query: str, runtime: ToolRuntime, k: int = 10) -> Command:
        """사용자가 이 Gem에 업로드한 파일 내용에서 관련 정보를 검색합니다."""
        results = await gem_file_rag.search(gem_id, query, k=k)
        if results is None:
            results = []
        if not results:
            msg = f"업로드된 파일에서 '{query}'와 관련된 내용을 찾지 못했습니다."
            return Command(update={
                "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
            })

        output_lines = [f"업로드 파일 검색 결과: '{query}'\n", "=" * 80]
        file_sources = []
        seen_files = set()
        for idx, r in enumerate(results, 1):
            output_lines.append(f"\n[파일 {idx}] {r['filename']} (유사도: {r['score']:.4f})")
            output_lines.append(f"\n내용:\n{r['text_chunk']}\n")
            output_lines.append("-" * 80)
            if r["filename"] not in seen_files:
                seen_files.add(r["filename"])
                snippet = " ".join((r["text_chunk"] or "").split())
                if len(snippet) > 160:
                    snippet = snippet[:160].rstrip() + "…"
                file_sources.append({
                    "type": "file",
                    "arxiv_id": "",
                    "title": r["filename"],
                    "summary": snippet,
                    "score": round(r["score"], 4),
                })
        return Command(update={
            "messages": [ToolMessage(content="\n".join(output_lines), tool_call_id=runtime.tool_call_id)],
            "sources": file_sources,
        })

    return search_gem_files

# 4. 에이전트 구조화 출력 포맷 스키마
class GemPaperRef(BaseModel):
    arxiv_id: str = Field(description="논문의 arXiv ID (예: 2504.10388)")
    title: str = Field(description="논문 제목")
    summary: str = Field(description="이 논문이 질문과 어떻게 관련되는지 한 문장 요약")

class GemAnswer(BaseModel):
    explanation: str = Field(description="질문에 대한 자연스러운 마크다운 설명")
    papers: list[GemPaperRef] = Field(default_factory=list, description="답변 근거가 된 논문 목록")

# 5. GemAgent 핵심 클래스 전체 구현
class GemAgent:
    def __init__(self, model: str = "openai:gpt-4o-mini"):
        self.model = model
        self.checkpointer = checkpointer

    def _build_system_prompt(
        self, db_sources: list[str], persona_prompt: str, has_files: bool = False, streaming: bool = False
    ) -> str:
        tool_lines = "\n".join(f"  · {_TOOL_CALL_DESC[src]}" for src in db_sources if src in _TOOL_CALL_DESC)
        if has_files:
            tool_lines += f"\n  · {_FILE_TOOL_DESC}"

        streaming_desc = "참고한 논문 목록을 본문에 나열하거나 별도 섹션을 만들지 마세요. 설명에만 집중합니다." if streaming else \
                         "papers에는 답변의 근거가 된 논문 각각을 정리합니다. 업로드 파일에서 얻은 내용의 경우 arxiv_id를 빈 문자열(\"\")로 설정합니다."

        return (
            f"{persona_prompt}\n\n"
            "작업 방식:\n"
            "- 질문 주제를 파악해서 아래 검색 도구 중 알맞은 것을 반드시 호출합니다.\n"
            f"{tool_lines}\n"
            "- 중요: 검색 도구에 전달하는 query는 반드시 영어로 작성하세요.\n"
            f"- {streaming_desc}\n"
            "- 답변은 항상 사용자가 질문한 언어로 작성합니다."
        )

    def _build_agent(self, db_sources: list[str], system_prompt: str, gem_id: str | None = None, has_files: bool = False):
        tools = [_TOOL_MAP[src] for src in db_sources if src in _TOOL_MAP]
        if not tools:
            tools = [search_bio_papers]
        if has_files and gem_id:
            tools.append(_make_file_search_tool(gem_id))

        full_system_prompt = self._build_system_prompt(db_sources, system_prompt, has_files)
        return create_agent(
            model=self.model,
            tools=tools,
            system_prompt=full_system_prompt,
            checkpointer=self.checkpointer,
            state_schema=GemAgentState,
            response_format=GemAnswer,
        )

    def _build_stream_agent(self, db_sources: list[str], system_prompt: str, gem_id: str | None = None, has_files: bool = False):
        tools = [_TOOL_MAP[src] for src in db_sources if src in _TOOL_MAP]
        if not tools:
            tools = [search_bio_papers]
        if has_files and gem_id:
            tools.append(_make_file_search_tool(gem_id))

        full_system_prompt = self._build_system_prompt(db_sources, system_prompt, has_files, streaming=True)
        return create_agent(
            model=self.model,
            tools=tools,
            system_prompt=full_system_prompt,
            checkpointer=self.checkpointer,
            state_schema=GemAgentState,
        )

    async def run(self, message: str, thread_id: str, db_sources: list[str], system_prompt: str, gem_id: str | None = None, has_files: bool = False) -> dict:
        agent = self._build_agent(db_sources, system_prompt, gem_id=gem_id, has_files=has_files)
        result = await agent.ainvoke(
            {"messages": [{"role": "user", "content": message}], "sources": []},
            {"configurable": {"thread_id": thread_id}},
        )
        structured = result.get("structured_response")
        if structured:
            answer = structured.explanation
            papers = [p.model_dump() for p in structured.papers]
        else:
            raw_content = result["messages"][-1].content
            try:
                parsed = json.loads(raw_content)
                answer = parsed.get("explanation", raw_content)
                papers = parsed.get("papers", [])
            except:
                answer = raw_content
                papers = []

        seen = set()
        unique_sources = []
        for s in result.get("sources", []):
            key = s.get("arxiv_id") or s.get("title", "")
            if key not in seen:
                seen.add(key)
                unique_sources.append(s)
        return {"answer": answer, "papers": papers, "sources": unique_sources}

    async def run_stream(
        self, message: str, thread_id: str, db_sources: list[str], system_prompt: str, gem_id: str | None = None, has_files: bool = False
    ) -> AsyncGenerator[dict, None]:
        agent = self._build_stream_agent(db_sources, system_prompt, gem_id=gem_id, has_files=has_files)
        _TOOL_STATUS = {
            "search_bio_papers": "paper_search",
            "search_cs_papers": "paper_search",
            "search_astronomy_papers": "paper_search",
            "search_gem_files": "file_search",
        }
        announced_tools = set()
        config = {"configurable": {"thread_id": thread_id}}
        captured_sources = []

        async for stream_mode, chunk in agent.astream(
            {"messages": [{"role": "user", "content": message}], "sources": []},
            config,
            stream_mode=cast(Any, ["messages", "values"]),
        ):
            if stream_mode == "values":
                if isinstance(chunk, dict):
                    sv = chunk.get("sources")
                    if sv:
                        captured_sources = list(sv)
                continue

            token, metadata = chunk
            names = []
            for tc in (getattr(token, "tool_call_chunks", None) or []):
                if tc.get("name"):
                    names.append(tc["name"])
            for tc in (getattr(token, "tool_calls", None) or []):
                if tc.get("name"):
                    names.append(tc["name"])
            for name in names:
                if name not in announced_tools:
                    announced_tools.add(name)
                    yield {"type": "status", "data": _TOOL_STATUS.get(name, "tool")}

            if isinstance(metadata, dict) and metadata.get("langgraph_node") == "tools":
                continue
            if getattr(token, "tool_calls", None):
                continue

            content = getattr(token, "content", "")
            if content:
                yield {"type": "token", "data": content}

        if captured_sources:
            seen = set()
            unique = []
            for s in captured_sources:
                key = s.get("arxiv_id") or s.get("title", "")
                if key and key not in seen:
                    seen.add(key)
                    unique.append(s)
            if unique:
                yield {"type": "papers", "data": unique}

print("✅ GemAgent 클래스 선언 완료!")

✅ GemAgent 클래스 선언 완료!


## 4단계. 구조화 응답 에이전트 실행

앞서 선언한 `GemAgent` 클래스를 생성하여, Pydantic DTO(`GemAnswer`) 구조에 맞게 단답형 답변을 얻는 시뮬레이션을 수행합니다.

In [20]:
async def run_structured_agent_demo():
    print("🚀 [Structured Agent] 구동 시작...")
    agent = GemAgent()
    res = await agent.run(
        message="Tell me about planetesimal formation and planetary migration.",
        thread_id="test-thread-gem-1",
        db_sources=["astronomy"],
        system_prompt="당신은 천체 물리학 연구 전문 어시스턴트입니다.",
        gem_id="gem-astronomy-1",
        has_files=False
    )
    print("\n✨ [실제 LLM 실행 결과]:")
    print(f"답변 (explanation):\n{res['answer']}")
    print(f"📚 인용 논문 수: {len(res['papers'])}개")
    for idx, p in enumerate(res['papers'], 1):
        print(f"  [{idx}] {p['title']} ({p['arxiv_id']})")

await run_structured_agent_demo()

Deserializing unregistered type __main__.GemAnswer from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'GemAnswer')]


🚀 [Structured Agent] 구동 시작...

✨ [실제 LLM 실행 결과]:
답변 (explanation):
Planetesimal formation and planetary migration are critical processes in the formation and evolution of planetary systems. 

**Planetesimal Formation:** This process involves the aggregation of dust and ice particles into larger bodies known as planetesimals, which are essential building blocks for planet formation. The formation typically occurs in protoplanetary disks, where particles face several challenges such as inefficient sticking, fragmentation, and radial drift. Recent models suggest that the growth of planetesimals may occur through mechanisms like mass transfer during high-speed collisions and fluffy growth, where materials with lower density have enhanced sticking probabilities.

Planetesimals can form in various locations within the disk, influenced by environmental conditions such as temperature and gas density. For instance, studies indicate that planetesimal formation could happen at the snowline (where

## 5단계. 실시간 스트리밍 대화(token, status, papers 이원화 구조)

구조화된 Pydantic 답변(`response_format=GemAnswer`)은 JSON이 완성될 때까지 한 글자씩 스트리밍으로 브라우저에 보여줄 수 없다는 단점이 있습니다.

이 문제를 해결하기 위해, 백엔드는 **스트리밍용 에이전트(response_format 없음)**를 별도로 사용합니다. 그리고 `astream()` API의 출력을 인터셉트하여 아래의 3가지 유형의 스트림 이벤트를 비동기적으로 내보내는 고급 아키텍처를 가집니다:

1. **`type: status`**: 에이전트가 RAG 검색 도구(`search_bio_papers` 등)를 호출하는 상태 인디케이터용 이벤트
2. **`type: token`**: 실시간 생성되는 답변 마크다운 텍스트 조각
3. **`type: papers`**: 최종 검색이 끝나 누적된 참고 문헌/업로드 파일 출처 목록

In [21]:
async def run_stream_parser_demo():
    print("🚀 [Stream Parser] 연결을 개시하고 데이터 수집을 시작합니다.\n")
    
    agent = GemAgent()
    stream_gen = agent.run_stream(
        message="What is the role of deep learning in CRISPR target design?",
        thread_id="test-stream-thread-999",
        db_sources=["bio"],
        system_prompt="당신은 유전공학 연구 전문 어시스턴트입니다.",
        gem_id="gem-bio-1",
        has_files=False
    )

    # 비동기 이벤트 스트림 파싱 루프
    async for event in stream_gen:
        ev_type = event.get("type")
        ev_data = event.get("data")
        
        if ev_type == "status":
            print(f"⚙️ [상태 알림] 에이전트가 현재 도구를 수행 중입니다: {ev_data}")
            
        elif ev_type == "token":
            print(ev_data, end="", flush=True)
            
        elif ev_type == "papers" and isinstance(ev_data, list):
            print("\n")
            print("📚 [최종 참고 문헌 목록 수신 완료]:")
            for idx, p in enumerate(ev_data, 1):
                print(f"  [{idx}] {p['title']} (Link: https://arxiv.org/abs/{p['arxiv_id']})")
                print(f"      요약: {p['summary']}")

await run_stream_parser_demo()

🚀 [Stream Parser] 연결을 개시하고 데이터 수집을 시작합니다.

⚙️ [상태 알림] 에이전트가 현재 도구를 수행 중입니다: paper_search
Deep learning has become an integral component of CRISPR target design, significantly enhancing the precision and effectiveness of genome editing. Here are the main roles of deep learning in this context:

1. **On-Target Prediction**: Deep learning models, such as DeepFM-Crispr, are designed to predict the on-target efficiency of CRISPR systems like Cas13. These models utilize comprehensive datasets, including evolutionary and structural information, to produce more accurate predictions of how well a specific guide RNA (gRNA) will bind to its target.

2. **Off-Target Effect Assessment**: A critical application of deep learning in CRISPR design is assessing potential off-target effects. By analyzing various genomic contexts, these models can identify which gRNAs might interact with unintended targets, thus improving the safety of CRISPR applications.

3. **Ensemble Learning Techniques**: Some approa

## 6단계. 백엔드 구현 파일 연계 정보

이 노트북에서 분석한 에이전트 동적 조립 및 스트리밍 전송 로직은 실제 백엔드의 다음 소스파일들에 구현되어 구동됩니다:

1. **`api.v1.gems.gem_agent`**
   - [gem_agent.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/gem_agent.py): `GemAgent` 클래스가 선언되어 있으며, `_build_agent()` 및 `_build_stream_agent()`를 통해 도구를 동적 구성하고 `run_stream` 비동기 제너레이터를 통해 `status`와 `token`, `papers` 이벤트를 나누어 클라이언트에 반환합니다.
2. **`api.v1.gems.services`**
   - [services.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/services.py): `GemService.chat_stream`에서 에이전트의 제너레이터를 받아 `fastapi.responses.StreamingResponse`로 래핑하여 SSE 형식을 준수해 전송하며, 대화가 안전하게 완료되면 비동기 백그라운드 태스크로 대화 이력과 세션을 DB에 마운트합니다.
3. **`api.v1.gems.endpoints`**
   - [endpoints.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/endpoints.py): `/gems/{gem_id}/chat` POST API를 통해 사용자의 대화 요청을 받아 실시간 SSE 스트림으로 클라이언트에 최종 라우팅합니다.